In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
base_url = "https://www.jobkorea.co.kr"
search_url = f"{base_url}/Search/?stext=데이터분석"
headers = {
    'User-Agent': 'Mozilla/5.0'
}

response = requests.get(search_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

In [3]:
# 공고 리스트 가져오기
articles = soup.find_all('article', class_='list-item')

# 데이터 저장용 리스트
company_list = []
recruit_list = []
detail_list = []
url_list = []

In [5]:
for article in articles:
    # 회사명
    company_tag = article.find('a', class_='corp-name-link dev-view')
    company = company_tag.text.strip() if company_tag else None

    # 채용공고명 dimension45
    recruit = article.get('data-gainfo')
    if recruit:
        if 'dimension45' in recruit:
            start = recruit.find('dimension45":"') + len('dimension45":"')
            end = recruit.find('"', start)
            recruit = recruit[start:end]
        else:
            recruit = "채용공고명 없음"
    else:
        recruit = "채용공고명 없음"

    # 상세 정보
    detail_tags = article.select('ul.chip-information-group li.chip-information-item')
    detail_str = "[" + ", ".join([li.get_text(strip=True) for li in article.select('ul.chip-information-group li.chip-information-item')]) + "]"


    # 채용공고 URL
    href = company_tag['href'] if company_tag and company_tag.has_attr('href') else None
    full_url = base_url + href if href else None

    # 리스트에 추가
    company_list.append(company)
    recruit_list.append(recruit)
    detail_list.append(detail_str)
    url_list.append(full_url)


In [12]:
df1 = pd.DataFrame({
    'Site': ['Job_Korea'] * len(company_list),
    'Col_Company': company_list,
    'Col_Recruit': recruit_list,
    'Col_detail': detail_list,
    'Col_url': url_list
})

# 결과 미리 보기
df1

,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Job_Korea,더치트주식회사,데이터 애널리스트 / 데이터분석 전문가 / 통계 전문가 / Data Analyst (통,"[경력무관, 대졸↑, 정규직 외, 서울 구로구, D-25]",https://www.jobkorea.co.kr/Recruit/GI_Read/466...
1,Job_Korea,넛지헬스케어㈜,[캐시워크-병역특례] 데이터분석 담당 산업기능요원,"[경력무관, 학력무관, 병역특례, 서울 강남구, D-8]",https://www.jobkorea.co.kr/Recruit/GI_Read/465...
2,Job_Korea,㈜메디젠휴먼케어,메디젠휴먼케어 데이터분석 담당 경력직 채용,"[경력3년↑, 석사↑, 정규직, 서울 송파구, D-42]",https://www.jobkorea.co.kr/Recruit/GI_Read/465...
3,Job_Korea,넛지헬스케어㈜,[캐시워크] 데이터분석 담당 채용전환형 인턴,"[신입, 대졸↑, 인턴, 서울 강남구, D-11]",https://www.jobkorea.co.kr/Recruit/GI_Read/465...
4,Job_Korea,한패스㈜,[한패스(주)/데이터분석팀] Google Analytics 데이터분석가,"[경력, 학력무관, 정규직, 서울 성동구, D-7]",https://www.jobkorea.co.kr/Recruit/GI_Read/465...
...,...,...,...,...,...
97,Job_Korea,None,채용공고명 없음,"[월급2,096,270원, 대전 유성구 구룡동, 등록일 3/24, 마감일 상시모집]",None
98,Job_Korea,None,채용공고명 없음,"[월급2,400,000원, 서울 서초구 양재동, 등록일 3/22, D-9]",None
99,Job_Korea,None,채용공고명 없음,"[월급2,585,360원, 경기 화성시, 등록일 3/20, D-1]",None
100,Job_Korea,None,채용공고명 없음,"[월급3,000,000원, 서울 송파구 삼전동, 등록일 3/18, 마감일 상시모집]",None


In [13]:
import os

os.makedirs("data_tmp", exist_ok=True)
df1.to_csv("data_tmp/data_jobkorea.csv", index=False, encoding="utf-8-sig")